# RetainIQ — Notebook 1: Exploratory Data Analysis

Raw data → univariate + bivariate analysis + statistical tests.

**Input:** `../data/telco_churn.csv`  
**Output:** Visual insights (no file written — preprocessing does the saving)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# ── VS Code: load raw CSV relative to this notebook's location ───────────────
DATA_PATH = os.path.join(os.path.dirname(os.path.abspath('__file__')), '..', 'data', 'telco_churn.csv')
df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()


In [ ]:
# ── Cell 2: Basic Inspection ─────────────────────────────────────────────────
print(df.dtypes)
print('\nNull values:\n', df.isnull().sum())
print('\nChurn distribution:\n', df['Churn'].value_counts())


In [ ]:
# ── Cell 3: df.info and duplicates ───────────────────────────────────────────
df.info()
df.describe()
print("Duplicates:", df.duplicated().sum())


In [ ]:
# ── Cell 4: Target Variable — Churn Rate ─────────────────────────────────────
churn_counts = df['Churn'].value_counts()
plt.figure(figsize=(6, 6))
plt.pie(churn_counts, labels=churn_counts.index, autopct='%1.1f%%',
        colors=['#66b3ff', '#ff9999'], startangle=140)
plt.title('Churn Distribution')
plt.tight_layout()
plt.show()


In [ ]:
# ── Cell 5: Numerical Distributions ──────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ['tenure', 'MonthlyCharges', 'TotalCharges']):
    ax.hist(df[col].dropna(), bins=30, color='steelblue', edgecolor='white')
    ax.set_title(f'Distribution of {col}')
    ax.set_xlabel(col)
    ax.set_ylabel('Count')
plt.tight_layout()
plt.show()


In [ ]:
# ── Cell 6: Categorical Distributions ────────────────────────────────────────
cat_cols = ['InternetService', 'Contract', 'PaymentMethod',
            'OnlineSecurity', 'TechSupport', 'StreamingTV']
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, col in zip(axes.flatten(), cat_cols):
    df[col].value_counts().plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
    ax.set_title(f'{col} Distribution')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()


In [ ]:
# ── Cell 7: Churn by Key Categorical Features ─────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ['Contract', 'InternetService', 'PaymentMethod']):
    ct = pd.crosstab(df[col], df['Churn'], normalize='index') * 100
    ct.plot(kind='bar', ax=ax, color=['#66b3ff', '#ff9999'], edgecolor='white')
    ax.set_title(f'Churn Rate by {col}')
    ax.set_ylabel('Percentage (%)')
    ax.tick_params(axis='x', rotation=30)
    ax.legend(['No Churn', 'Churn'])
plt.tight_layout()
plt.show()


In [ ]:
# ── Cell 8: Statistical Test — Monthly Charges vs Churn (Welch's t-test) ─────
from scipy import stats

churned     = df[df['Churn'] == 'Yes']['MonthlyCharges']
not_churned = df[df['Churn'] == 'No']['MonthlyCharges']

t_stat, p_value = stats.ttest_ind(churned, not_churned, equal_var=False)
print(f't-statistic : {t_stat:.4f}')
print(f'p-value     : {p_value:.6f}')
print('Conclusion  :', 'Significant difference (p < 0.05)' if p_value < 0.05 else 'No significant difference')
